<a href="https://colab.research.google.com/github/nilnil47/simple-committee-machine/blob/main/simple_commette_machine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Minimal Teacher-Student Model (Knowledge Distillation) Implementation

Knowledge Distillation involves training a smaller 'Student' model to mimic the behavior of a pre-trained, larger 'Teacher' model.

### Teacher: Hermite Polynomial $He_3(w_* \cdot x)$

In this setup, the teacher is not a neural network but a fixed direction $w_*$. The target output is the third Hermite polynomial $He_3(z) = z^3 - 3z$ applied to the projection of $x$ onto $w_*$.

In [22]:
import torch
import torch.nn as nn
import torch.optim as optim
import math

# Parameters
dimension = 10
batch_size = 64
n_hidden = 32 # N

# 1. Define the Teacher (Fixed Direction w_*)
w_star = torch.randn(dimension, 1)
w_star = w_star / torch.norm(w_star) # Normalize

def hermite_teacher(x, w):
    z = torch.matmul(x, w)
    # He_3(z) = z^3 - 3z
    return (z**3 - 3*z).squeeze()

# 2. Define the Committee Student Model
class CommitteeStudent(nn.Module):
    def __init__(self, d: int, n_hidden: int) -> None:
        super().__init__()
        self.readout_scale = 1.0 / math.sqrt(n_hidden)
        self.W = nn.Parameter(torch.empty(n_hidden, d))
        nn.init.normal_(self.W, mean=0.0, std=1.0 / math.sqrt(d))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # f(x; W) = (1/sqrt(N)) sum_i erf(w_i · x)
        h = torch.erf(x @ self.W.T)
        return self.readout_scale * h.sum(dim=-1)

student_hermite = CommitteeStudent(dimension, n_hidden)
optimizer = optim.Adam(student_hermite.parameters(), lr=0.0001)
criterion = nn.MSELoss()

In [23]:
# 3. Training Loop
for epoch in range(1000):
    x_train = torch.randn(batch_size, dimension)
    with torch.no_grad():
        y_teacher = hermite_teacher(x_train, w_star)

    y_student = student_hermite(x_train)
    loss = criterion(y_student, y_teacher)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/500], Loss: {loss.item():.4f}")

Epoch [100/500], Loss: 2.3902
Epoch [200/500], Loss: 13.4503
Epoch [300/500], Loss: 2.8897
Epoch [400/500], Loss: 19.4815
Epoch [500/500], Loss: 7.7674
Epoch [600/500], Loss: 2.0371
Epoch [700/500], Loss: 24.1908
Epoch [800/500], Loss: 1.9218
Epoch [900/500], Loss: 7.4346
Epoch [1000/500], Loss: 22.5397


In [14]:
### Evaluate on Test Loss
# Generate fresh data that the model hasn't seen
x_test = torch.randn(1000, dimension)
with torch.no_grad():
    y_test_teacher = hermite_teacher(x_test, w_star)
    y_test_student = student_hermite(x_test)
    test_loss = criterion(y_test_student, y_test_teacher)

print(f"Final Training Loss: {loss.item():.4f}")
print(f"Test Loss (Generalization): {test_loss.item():.4f}")

Final Training Loss: 6.4343
Test Loss (Generalization): 3.6374


### Training with Langevin Dynamics
To implement Langevin Dynamics, we manually update the weights by adding Gaussian noise scaled by the learning rate and a temperature parameter $\beta^{-1}$ after the standard gradient calculation.

In [13]:
# Reset student for Langevin training
student_langevin = CommitteeStudent(dimension, n_hidden)
learning_rate = 0.01
beta_inv = 0.001 # Noise scale (Inverse temperature)

for epoch in range(1000):
    x_train = torch.randn(batch_size, dimension)
    y_teacher = hermite_teacher(x_train, w_star)

    y_student = student_langevin(x_train)
    loss = criterion(y_student, y_teacher)

    student_langevin.zero_grad()
    loss.backward()

    # Langevin Update: w = w - lr * grad + sqrt(2 * lr * beta_inv) * noise
    with torch.no_grad():
        for param in student_langevin.parameters():
            noise = torch.randn_like(param) * math.sqrt(2 * learning_rate * beta_inv)
            param.add_(param.grad, alpha=-learning_rate)
            param.add_(noise)

    if (epoch + 1) % 100 == 0:
        print(f"Langevin Epoch [{epoch+1}/1000], Loss: {loss.item():.4f}")

Langevin Epoch [100/1000], Loss: 6.7434
Langevin Epoch [200/1000], Loss: 2.5294
Langevin Epoch [300/1000], Loss: 3.8325
Langevin Epoch [400/1000], Loss: 3.0899
Langevin Epoch [500/1000], Loss: 4.0058
Langevin Epoch [600/1000], Loss: 5.4420
Langevin Epoch [700/1000], Loss: 1.4367
Langevin Epoch [800/1000], Loss: 3.2843
Langevin Epoch [900/1000], Loss: 4.5504
Langevin Epoch [1000/1000], Loss: 6.4343


### Version Control in Colab

*   **Internal History**: `File` > `Revision history` (Ctrl+Alt+Shift+H).
*   **GitHub**: `File` > `Save a copy in GitHub` to push your current notebook to a repository.
*   **Manual Git**: You can use the cell below to check your git configuration if you are working within a cloned repo.

In [24]:
!git --version

git version 2.34.1
